# Visualizing Data in Roman ASDF Files with `matplotlib`

***

## Kernel Information and Read-Only Status

To run this notebook, please select the "Roman Research Nexus" kernel at the top right of your window.

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.

## Introduction
This tutorial demonstrates how to visualize and explore Roman WFI image data arrays by creating static images with world coordinate system (WCS) overlays in `matplotlib`. A companion tutorial shows [how to use the Jdaviz tool to interactively explore WFI image data](LINK-TO-COMPANION-NOTEBOOK-IF-DESIRED).

We focus on how to visualize WFI Level 2 (L2; calibrated rate image) data in ASDF format. For WFI, L2 indicates that the data have been processed to flag and/or correct for detector-level effects (e.g., saturation, classic non-linearity, etc.) and the resultants fitted into a count rate image. Each L2 ASDF file contains a single WFI detector, thus a complete WFI exposure is made up of 18 L2 files. For more information WFI L2 files, please see the RDox article on [Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-Level2-CalibratedExposuresLevel2).

## Imports
- *astropy.visualization.simple_norm* for automatically scaling image arrays
- *astropy.wcs.WCS* to create Astropy WCS objects
- *matplotlib.pyplot* for creating static image previews
- *numpy* for array calculations and manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *s3fs* to access data in an AWS S3 bucket

In [ ]:
%matplotlib inline
from astropy.visualization import simple_norm
# from astropy.coordinates import SkyCoord
# from astropy.table import Table
from astropy.wcs import WCS
# import copy
import matplotlib.pyplot as plt
import numpy as np

# from jdaviz import Imviz
import roman_datamodels as rdm
# import asdf
import s3fs
# import time

## Loading data
We use the same data file in both this notebook and [the Jdaviz notebook](ANOTHER-LINK-TO-POTENTIALLY-FILL-IN). Note that the static image portions can be used on any `numpy.ndarray` object, and the WCS axes may be optionally included with any `astropy.wcs.WCS` object.

A complete explanation on how to load and work with Roman ASDF files is provided in the notebook tutorial [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb). We read in the data using the `roman_datamodels` package:

In [ ]:
asdf_dir_uri = 's3://stpubdata/roman/nexus/soc_simulations/tutorial_data/'
fs = s3fs.S3FileSystem(anon=True)

asdf_file_uri = asdf_dir_uri + 'r0003201001001001004_0001_wfi01_f106_cal.asdf'
file = rdm.open(fs.open(asdf_file_uri, 'rb'))

## Static Image Display

### Plot an Image with Dynamic Scaling

Here we show how to use matplotlib and Astropy to plot the data array from our WFI image and scale the array automatically.

In [ ]:
# Set the image normalization. Here we use a inverse hyperbolic sine scale 
# with the minimum and maximum of the range specified as 0.5 to 4 DN/s.
# These limits can be adjusted based on examination of the pixel values
# (e.g., a histogram).
norm = simple_norm(file.data, 'asinh', vmin=0.5, vmax=4)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.imshow(file.data, norm=norm, origin='lower')
ax.set_xlabel('X Science Axis (pixels)')
ax.set_ylabel('Y Science Axis (pixels)')
ax.set_title('Roman I-Sim Simulation WFI01')
plt.colorbar(sc, ax=ax)
plt.tight_layout();

We can see a bright, extended source in the right portion of the image. We isolate and examine that region a little more closely. Based on the image above, let's isolate science Y coordinates 2000 – 2500 and science X coordinates 3500 - 4000.

In [ ]:
# Close the previous figure
plt.close()

# Make new figure zoomed in on 1750 <= X <= 2250 and 500 <= Y <= 1000:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.imshow(file.data, norm=norm, origin='lower')
ax.set_xlabel('X Science Axis (pixels)')
ax.set_ylabel('Y Science Axis (pixels)')
ax.set_title('Roman I-Sim Simulation (Data) WFI01')
ax.set_xlim(3500, 4088)
ax.set_ylim(2000, 2588)
plt.colorbar(sc, ax=ax)
plt.tight_layout();

We can see some strange features in the right side of this image. They were also in the previous plot, but were not as evident at the previous level of zoom. These features are not some interesting new science, but are actually instrumental artifacts, and we can see in the data quality (DQ) array that they are marked for our reference in any analysis we may perform. Not all artifacts or DQ flags are necessarily bad for analysis, but here we show all pixels with DQ values > 0 (any quality flags; white) compared to pixels with DQ values equal to 0 (good; black). For more information on how to work with DQ flags, see the [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) tutorial. The [Detector Performance](https://roman-docs.stsci.edu/roman-instruments-home/wfi-imaging-mode-user-guide/wfi-detectors/detector-performance) article and the articles under it will provide detail on instrumental artifacts and detector performance. Please note that several analyses are still ongoing and RDox will be updated in the future.

In [ ]:
# Close the previous figure
plt.close()

# Display the data quality (DQ) array for the same region.
# Convert the DQ values to boolean True (bad) and False (good) for simple display.
# The binary colormap normally goes from white to black, but we have inverted it
# using binary_r to ease visibility of flagged pixels.
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(np.bool(file.dq), origin='lower', cmap='binary_r')
ax.set_xlabel('X Science Axis (pixels)')
ax.set_ylabel('Y Science Axis (pixels)')
ax.set_title('Roman I-Sim Simulation (DQ) WFI01')
ax.set_xlim(3500, 4088)
ax.set_ylim(2000, 2588)
plt.tight_layout();

If we want to add information about the sky coordinates rather than displaying the image in pixels, we can use the `gwcs.wcs.WCS` object in the file metadata. To overlay the coordinate grid, however, we will need to transform the WCS to an `astropy.wcs.WCS` object, which needs a FITS Simple Imaging Polynomial (SIP) representation of the WCS. 

In [ ]:
# Close the previous figure
plt.close()

# Set the matplotlib backend (change this to "%matplotlib widget" for interactive plots)
%matplotlib inline

fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': WCS(file.meta.wcs.to_fits_sip())})
sc = ax.imshow(file.data, norm=norm, origin='lower')
ax.imshow(file.data, norm=norm, origin='lower')
plt.colorbar(sc, ax=ax)
ax.grid(color='white', alpha=0.3)
ax.set_xlabel('Right Ascension (deg)')
ax.set_ylabel('Declination (deg)')
plt.show()

In [ ]:
# Close figures before proceeding
plt.close('all')

We can also use `matplotlib` to examine the image interactively using the `%matplotlib widget` magic command. To do this, try re-running the cell above, but replacing `%matplotlib inline` with `%matplotlib widget`. If you use this command, you will see some basic icons on the left of the image that allow you to pan, zoom, return to the original display, and save the current view to a file. Notice that the WCS gridlines will adjust for different levels of zoom. You can do this for any of the `matplotlib` plots above.

For even more interactivity, continue to [the Jdaviz notebook](ANOTHER-LINK-TO-POTENTIALLY-ADD).

## Additional Resources

- [RDox WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-L2ScienceDataSpecifications)

## About this Notebook
**Author:** Tyler Desjardins, Brett Morris, R. Diaz  
**Updated On:** 2025-12-13

<table width="100%" style="border:none; border-collapse:collapse;">
  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>